In [ ]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
import cvxpy as cp
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list, optimal_leaf_ordering, fcluster
from sklearn.covariance import LedoitWolf

In [ ]:
@dataclass
class Universe:
  Bonds:List[str]
  ManagedFutures:List[str]
  Commodities:List[str]
  High_Beta:List[str]
  High_Yield:List[str]
  Sat_Defensive:List[str]

@dataclass
class HERCParams:
  quantile:float|None=None
  use_lw_shrinkage:bool=False
  k_max:int=5


In [ ]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers_raw = list(asdict(universe).values())
    tmp = []
    for t in tickers_raw:
      if isinstance(t, list):
        tmp.extend(t)
      elif isinstance(t, str):
        tmp.append(t)

      else:
        print(f"Warning: Skipping {t} | type: {type(t)} ")

    tickers_clean = list(set(tmp))

    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      if benchmark not in tickers_clean:
        tickers_clean.append(benchmark)

      df = yf.download(tickers_clean, start, end, interval)["Close"]

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    self.universe = data_raw.columns

    return data_raw, benchmark


  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [ ]:
class Filter:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      **kwargs
    )

In [ ]:
class HERCOptimizer:
  def __init__(self, optim_params:HERCParams, debug:bool=False, **kwargs):
    self.debug = debug
    self.p = optim_params

  def compute_exp_disp(self, dist):
    return

  def compute_W_k(self, dist_mtx, cluster_labels):
    w_k = 0
    unique_clusters = np.unique(cluster_labels)

    for c_id in unique_clusters:
      cluster_indices = np.where(cluster_labels == c_id)[0]
      C_r = len(cluster_indices)

      sub_dist = v[np.ix_(cluster_indices, cluster_indices)]

      D_r = np.sum(sub_dist ** 2)

      W_k += (1.0 / (2.0 * C_r)) * D_r

    return W_k

  def get_k_clusters(self, dist, Z):
    dist_mtx = squareform(dist)

    n_assets = len(dist_mtx)
    if self.p.k_max > n_assets:
      raise ValueError(f"k_max {self.p.k_max} > n of distances {len(dist)}")

    gap_k_list = []

    for k in range(1, self.p.max_k):
      clusters = fcluster(Z, t=k, criterion='maxclust')

      W_k = self.compute_W_k(dist_mtx, clusters)

      ex_disp, s_k = self.compute_exp_disp(dist, k)

      obs_disp = np.log(W_k)
      gap_k = ex_disp - obs_disp

      print(s_k*np.sqrt(1 + 1/self.p.n_sim))
      gap_k_list.append(gap_k)



  def get_clusters(self, returns: pd.DataFrame) -> pd.DataFrame:
    corr = returns.corr().clip(-1.0, 1.0)

    dist = np.sqrt(2 * (1 - corr))
    np.fill_diagonal(dist.values, 0)
    comp_dist = squareform(dist.values)

    Z = linkage(comp_dist, method="ward")
    Z_opt = optimal_leaf_ordering(Z, comp_dist)

    reordered_indices = leaves_list(Z_opt)
    seriated_corr = corr.iloc[reordered_indices, reordered_indices]

    k = self.get_k_clusters(comp_dist)

    return k



  def cvar_optimizer(self, returns, clusters):
    pass

  def _apply_ledoit_wolf(self, returns):
    model_fit = LedoitWolf().fit(returns)
    return model_fit.covariance_, model_fit.shrinkage_

  def vol_optimizer(self, returns):
    k = self.get_clusters(returns)
    cov, shrinkage = self._apply_ledoit_wolf(returns)




  def optimize_w(self, returns):
    if self.p.quantile is not None:
      return self.cvar_optimizer(returns)
    else:
      return self.vol_optimizer(returns)

In [ ]:
class Portfolio(DataStore, Filter, HERCOptimizer):
  def __init__(self, optim_params, debug:bool=False, **kwargs):
    super().__init__(
      debug=debug,
      optim_params=optim_params,
      **kwargs
    )

  def get_data(self, universe, start, end):
    data, benchmark = ptf._get_data(
      universe=universe,
      start=start,
      end=end
    )

    returns = data.pct_change().dropna()
    benchmark = benchmark.pct_change().dropna()

    return returns, benchmark

  def optimize(self, returns):
    self.optimize_w(returns)




In [135]:
optim_params = HERCParams()

In [138]:
ptf = Portfolio(optim_params=optim_params, debug=True)
returns, benchmark = ptf.get_data(
    universe=test_universe,
    start="2021-01-01",
    end="2026-01-01"
)